In [1]:
# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
import re
import warnings

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings('ignore')

plt.style.use('ggplot')


# =========================
# LOAD DATASET
# =========================

df = pd.read_csv("../data/raw/raw_analyst_ratings.csv")

print("Dataset Loaded Successfully")
print(df.head())


# =========================
# BASIC DATA INSPECTION
# =========================

print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nDataset Info:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())


# =========================
# CONVERT DATE COLUMN
# =========================

df['date'] = pd.to_datetime(df['date'], errors='coerce')

print("\nDate Conversion Completed")


# =========================
# HEADLINE LENGTH ANALYSIS
# =========================

df['headline_length'] = df['headline'].astype(str).apply(len)

print("\nHeadline Length Statistics:")
print(df['headline_length'].describe())


# =========================
# VISUALIZATION:
# HEADLINE LENGTH DISTRIBUTION
# =========================

plt.figure(figsize=(10,6))

sns.histplot(df['headline_length'], bins=50)

plt.title("Distribution of Headline Lengths")
plt.xlabel("Headline Length")
plt.ylabel("Frequency")

plt.show()


# =========================
# ARTICLES PER PUBLISHER
# =========================

publisher_counts = df['publisher'].value_counts()

print("\nTop Publishers:")
print(publisher_counts.head(10))


# =========================
# VISUALIZATION:
# TOP PUBLISHERS
# =========================

top_publishers = publisher_counts.head(10)

plt.figure(figsize=(12,6))

sns.barplot(
    x=top_publishers.values,
    y=top_publishers.index
)

plt.title("Top 10 Publishers by Article Count")
plt.xlabel("Number of Articles")
plt.ylabel("Publisher")

plt.show()


# =========================
# PUBLICATION TREND ANALYSIS
# =========================

df['day'] = df['date'].dt.date

daily_articles = df.groupby('day').size()

print("\nDaily Article Counts:")
print(daily_articles.head())


# =========================
# VISUALIZATION:
# DAILY NEWS VOLUME
# =========================

plt.figure(figsize=(15,6))

daily_articles.plot()

plt.title("Daily News Volume Over Time")
plt.xlabel("Date")
plt.ylabel("Number of Articles")

plt.show()


# =========================
# ROLLING TREND ANALYSIS
# =========================

rolling_news = daily_articles.rolling(window=7).mean()

plt.figure(figsize=(15,6))

plt.plot(
    daily_articles.index,
    daily_articles.values,
    alpha=0.5,
    label='Daily Volume'
)

plt.plot(
    rolling_news.index,
    rolling_news.values,
    label='7-Day Rolling Average'
)

plt.title("News Volume Trend Over Time")
plt.xlabel("Date")
plt.ylabel("Article Count")

plt.legend()

plt.show()


# =========================
# PUBLISHING TIME ANALYSIS
# =========================

df['hour'] = df['date'].dt.hour

hourly_counts = df['hour'].value_counts().sort_index()

print("\nHourly Publishing Counts:")
print(hourly_counts)


# =========================
# VISUALIZATION:
# PUBLISHING TIMES
# =========================

plt.figure(figsize=(12,6))

sns.lineplot(
    x=hourly_counts.index,
    y=hourly_counts.values
)

plt.title("News Publishing Frequency by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Articles")

plt.show()


# =========================
# TEXT CLEANING
# =========================

text = " ".join(df['headline'].astype(str))

words = re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())

filtered_words = [
    word for word in words
    if word not in ENGLISH_STOP_WORDS
]


# =========================
# COMMON KEYWORDS
# =========================

word_counts = Counter(filtered_words)

print("\nMost Common Keywords:")
print(word_counts.most_common(20))


# =========================
# VISUALIZATION:
# COMMON KEYWORDS
# =========================

common_words = pd.DataFrame(
    word_counts.most_common(15),
    columns=['word', 'count']
)

plt.figure(figsize=(12,6))

sns.barplot(
    x='count',
    y='word',
    data=common_words
)

plt.title("Most Common Keywords in Headlines")

plt.show()


# =========================
# TF-IDF ANALYSIS
# =========================

tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=20
)

X = tfidf.fit_transform(df['headline'].astype(str))

keywords = tfidf.get_feature_names_out()

print("\nTop TF-IDF Keywords:")
print(keywords)


# =========================
# TOPIC MODELING WITH LDA
# =========================

vectorizer = CountVectorizer(
    stop_words='english',
    max_df=0.95,
    min_df=2
)

X_counts = vectorizer.fit_transform(
    df['headline'].astype(str)
)

lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42
)

lda.fit(X_counts)

words = vectorizer.get_feature_names_out()

print("\nLDA Topics:")

for i, topic in enumerate(lda.components_):

    print(f"\nTopic {i+1}")

    top_words = [
        words[j]
        for j in topic.argsort()[-10:]
    ]

    print(top_words)


# =========================
# EMAIL DOMAIN EXTRACTION
# =========================

def extract_domain(publisher):

    match = re.search(
        r'@([\w.-]+)',
        str(publisher)
    )

    if match:
        return match.group(1)

    return None


df['domain'] = df['publisher'].apply(extract_domain)

print("\nTop Domains:")
print(df['domain'].value_counts().head(10))


# =========================
# VISUALIZATION:
# TOP DOMAINS
# =========================

top_domains = df['domain'].value_counts().head(10)

plt.figure(figsize=(12,6))

sns.barplot(
    x=top_domains.values,
    y=top_domains.index
)

plt.title("Top Publisher Domains")

plt.show()


# =========================
# INITIAL TASK 2 PROGRESS
# =========================

print("\nInitial Task 2 Progress:")
print("Technical indicator implementation will follow using stock price data.")


# =========================
# FINAL SUMMARY
# =========================

print("\nEDA COMPLETED SUCCESSFULLY")

print("""
Key Findings:
- Headline lengths were analyzed
- Top publishers identified
- Publication trends explored
- Common keywords extracted
- Topic modeling performed
- Publishing time patterns analyzed
- Publisher domains extracted
""")

ModuleNotFoundError: No module named 'pandas'

# Publisher Analysis

This section identifies the most active financial news publishers
and analyzes their contribution patterns.

In [ ]:
# =========================
# TASK 2: STOCK PRICE ANALYSIS (MULTIPLE STOCKS)
# =========================

import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# 1. LOAD ALL STOCK DATA
# -------------------------

aapl = pd.read_csv("../data/raw/AAPL.csv")
meta = pd.read_csv("../data/raw/META.csv")
goog = pd.read_csv("../data/raw/GOOG.csv")
nvda = pd.read_csv("../data/raw/NVDA.csv")
amzn = pd.read_csv("../data/raw/AMZN.csv")

print("All datasets loaded successfully")


# -------------------------
# 2. CONVERT DATE COLUMN
# -------------------------

for df in [aapl, meta, goog, nvda, amzn]:
    df['Date'] = pd.to_datetime(df['Date'])
    df.sort_values('Date', inplace=True)
    df.reset_index(drop=True, inplace=True)


# -------------------------
# 3. TECHNICAL INDICATOR
# SIMPLE MOVING AVERAGE (20-DAY)
# -------------------------

for df in [aapl, meta, goog, nvda, amzn]:
    df['SMA_20'] = df['Close'].rolling(window=20).mean()


# -------------------------
# 4. VISUALIZATION (ONE STOCK FOR INTERIM)
# Using NVIDIA as example
# -------------------------

plt.figure(figsize=(14,7))

plt.plot(nvda['Date'], nvda['Close'], label='NVDA Close Price')
plt.plot(nvda['Date'], nvda['SMA_20'], label='NVDA 20-Day SMA')

plt.title("NVIDIA Stock Price with 20-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("Price")

plt.legend()
plt.show()


# -------------------------
# 5. QUICK INSIGHT OUTPUT
# -------------------------

print("""
TASK 2 INITIAL ANALYSIS:

Stock price data for five major tech companies (AAPL, META, GOOG, NVDA, AMZN)
was loaded from CSV files.

A 20-day Simple Moving Average (SMA) was applied to smooth price fluctuations
and identify underlying trend direction.

NVIDIA was used as a representative example for visualization.

This analysis prepares the dataset for later correlation with financial news sentiment.
""")

ModuleNotFoundError: No module named 'pandas'